In [1]:
%load_ext autoreload
%autoreload 3

In [2]:
from __future__ import annotations

import math
import numpy as np
from typing import Any, Optional

from minitorch.tensor.tensor import Tensor
from minitorch.attention.attention import MultiHeadAttention
from minitorch.tokenization.tokenizer import CharTokenizer, BPETokenizer
from minitorch.dataloaders.dataloader import DataLoader
from minitorch.embendding.embed import EmbeddingLayer, Embedding, PositionalEncoding
from minitorch.losses.losses import MSE, SoftMaxCrossEntropy, BCEWithLogits, log_softmax
from minitorch.nn.layers import Linear, Module, LayerNormalization
from minitorch.activations.activations import Softmax, GELU
from minitorch.optimizers.optim import AdamW

In [3]:
text_path = 'C:\\Users\\User\\Desktop\\babytorch\\datasets\\BIASHARA_Cleaned.txt'

with open(text_path, 'r', encoding='utf-8') as f:
    text = f.read()

In [18]:
#* split into training and val sets
n = 0.95
text_len = len(text)

chars = sorted(set(text))

context_length, batch_size, embed_dim = 8,8,32
#* tokenize the text
tokenizer = CharTokenizer() #* character level tokenization
tokenizer.build_vocab(chars)
vocab_size = tokenizer.vocab_size

In [19]:
data = Tensor(tokenizer.encode(text))
train_data = data[:int(n*data.shape[0])]
val_data = data[int(n*data.shape[0]):]
train_loader = DataLoader(train_data, context_length)
val_loader = DataLoader(val_data, context_length)
train_xs, train_ys = train_loader.get_batch(batch_size)
val_xs, val_ys = val_loader.get_batch(batch_size)

In [20]:
train_xs.requires_grad = True
train_ys.requires_grad = True

In [21]:
token_embedding = Embedding(vocab_size, embed_dim)
pos_encoding = PositionalEncoding(vocab_size, embed_dim)
embeddings = token_embedding(train_xs)
pos_encodings = pos_encoding(embeddings)
tokens_embeddings = embeddings + pos_encodings
tokens_embeddings.shape

(8, 8, 32)

In [22]:
from typing import Any


from minitorch.nn.layers import Parameter
from minitorch.tensor.tensor import Tensor


loss_fn = SoftMaxCrossEntropy()
softmax = Softmax()


class Mha(Module):
    def __init__(self, embed_dim: int, n_heads: int, dim:int) -> None:
        super().__init__()
        self.mha = MultiHeadAttention(embed_dim, n_heads)
        self.layer_norm = LayerNormalization(dim)
        
    def __call__(self, input: Tensor) -> Tensor:
        att, _  = self.mha(self.layer_norm(input))
        return input + att
    
    def parameters(self) -> list[Parameter]:
        params = []
        params.extend(self.mha.parameters())
        params.extend(self.layer_norm.parameters())
        return params

class Block(Module):
    def __init__(self,embed_dim, n_heads, dim) -> None:
        super().__init__()
        self.mha = Mha(embed_dim, n_heads, dim)
        self.ffd = Linear(embed_dim, embed_dim)
        self.layer_norm = LayerNormalization(embed_dim)

    def __call__(self, input: Tensor) -> Tensor:
        x = self.mha(input)
        x = self.mha(input)
        x = self.layer_norm(x)
        x = x + self.ffd(x)
        return x 
    
    def parameters(self) -> list[Parameter]:
        params = []
        params.extend(self.mha.parameters())
        params.extend(self.ffd.parameters())
        return params
    
class GPT(Module):
    def __init__(self, vocab_size:int, embed_dim:int, max_seq_len:int,
                n_heads:int,n_blocks: int) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len
        
        self.embedding_table = Embedding(vocab_size,embed_dim)
        self.position_table = PositionalEncoding(max_seq_len, embed_dim)
        self.transformer_blocks = [
            Block(embed_dim, n_heads, self.embed_dim)
            for _ in range(n_blocks)
        ]
        self.lm_head = Linear(embed_dim, vocab_size)

    def __call__(self,idx: Tensor, targets:Tensor|None = None) -> tuple[Tensor, Tensor|None]:
        token_embed = self.embedding_table(idx)
        pos_encoding = self.position_table(token_embed)
        x = token_embed + pos_encoding
        
        for block in self.transformer_blocks:
            x = block(x)
            
        logits = self.lm_head(x)
        loss = None
        if targets:
            loss = loss_fn(logits, targets)
            return logits, loss
        return logits, loss
    
    def generate(self, idx:Tensor, max_new_tokens: int):
        for _ in range(max_new_tokens):
            logits,_ = self(idx)
            logits = logits[:,-1,:]
            probs = softmax(logits)
            next_token = np.array([
                        np.random.choice(probs.data.shape[1], p=row)
                        for row in probs.data
                        ])[:, None]
            idx = Tensor(np.concatenate([idx.data, next_token], axis=1))
        return idx
    
    def parameters(self):
        params = []
        params.extend(self.embedding_table.parameters())
        params.extend(self.position_table.parameters())
        params.extend(block.parameters()[0] for block in self.transformer_blocks)
        params.extend(self.lm_head.parameters())
        return params

In [9]:
def evaluate_loss(
    train_set: tuple[Tensor],
    val_set: tuple[Tensor],
    model: GPT,
    eval_iters: int) -> dict[str, Any]:
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = np.zeros(shape=(eval_iters))
        for i in range(eval_iters):
            if split == 'train':
                _, loss = model(*train_set)
                losses[i] = loss.data
            else:
                _, loss = model(*val_set)
                losses[i] = loss.data
        out[split] = losses.mean()
    model.train()
    return out

In [23]:
###############################################################
#* define the hyperparameters
learning_rate = 1e-4
max_seq_len = 100
n_heads = 8
max_iters = 1000
eval_iters = 100

#* ###########################################################
gpt = GPT(vocab_size, embed_dim,max_seq_len, n_heads, n_blocks=6) #* model
params = gpt.parameters()                         #* parameters
optimizer = AdamW(params, lr=learning_rate)                #* optimizer

#* ###########################################################
#* training loop
print("Epoch        | Training Loss         | Validation Loss | ")
print("-" * 60)

for i in range(max_iters):
    optimizer.zero_grad()
    if (i) % eval_iters == 0:
        losses = evaluate_loss((train_xs,train_ys), (val_xs, val_ys), gpt, eval_iters)
        print(f'{i+1}         | {losses["train"]:4f}        | {losses["val"]:4f}')
        
    _, loss = gpt(train_xs, train_ys)
    loss.backward()
    
    optimizer.step()
    

Epoch        | Training Loss         | Validation Loss | 
------------------------------------------------------------
1         | 3.845571        | 3.846499
101         | 3.507798        | 3.556231
201         | 3.250663        | 3.347722
301         | 3.063661        | 3.214255
401         | 2.928113        | 3.136910
501         | 2.826740        | 3.096375
601         | 2.747502        | 3.078696
701         | 2.683096        | 3.075290
801         | 2.629271        | 3.080828
901         | 2.583440        | 3.091748


In [16]:
############################################################################

#* Try generating using the trained model
# alpha_index = tokenizer.char_to_id[chars[2]] #* first real alphabet index --->(a)
# start_point = Tensor.randint(alpha_index,vocab_size, shape=(1,1))
idx = Tensor.ones((1,1), requires_grad=True) 

#* generated text
generated_text = tokenizer.decode(gpt.generate(idx, 50).data.astype(int).tolist()[0])
generated_text

'<EOT>rzuwjbeiewwesihehevs<EOT>korbe<SOT>cbiixisozmzsbbrrxbtiees'

In [12]:
tokenizer.vocabulary

dict_keys(['<UNK>', '<EOT>', '<SOT>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z'])

In [13]:
params = []
params.extend([1])
params.extend([2])
params

[1, 2]